# 01-1. AI App, Workflow, Agent 비교

- 핵심 기술: 일반 LLM 호출, 고정 Workflow, LLM 기반 1회성 Tool 선택(입문형 Agent)

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. AI App, Workflow, Agent의 실행 구조 차이를 설명할 수 있다.
2. 동일한 사용자 요청을 세 가지 방식으로 각각 구현할 수 있다.
3. 각 방식의 실행 로그를 비교하여 자율성과 통제 가능성의 차이를 확인할 수 있다.
4. Agent가 Workflow보다 항상 우수하지는 않은 이유를 Tool 선택 실패 시나리오로 설명할 수 있다.

## 2. 문제 상황

사용자가 다음과 같이 요청했다고 하자.

> "입력된 문서에서 핵심 키워드를 추출하고 요약해줘."

이 요청은 다음 세 가지 방식으로 각각 구현할 수 있다.

- **AI App**: LLM을 한 번 호출해서 바로 답변을 받는다.
- **Workflow**: 키워드 추출 → 요약의 순서를 개발자가 미리 정해 놓고 그대로 실행한다.
- **Agent**: 이 요청을 처리하는 데 어떤 Tool이 필요한지 LLM이 직접 판단한 뒤 실행한다.

>세 방식은 결과물이 비슷해 보일 수 있지만, 실행 경로를 결정하는 주체와 통제 가능성이 다르다.  
>이 차이를 이해하지 못하면, 간단한 요청에도 불필요하게 복잡한 Agent 구조를 적용하거나  
>반대로 유연성이 필요한 상황에 고정된 Workflow만 사용하는 설계 오류가 발생한다.  

## 3. 핵심 개념

### 3.1 개념 정의

**AI App**은 사용자 입력을 받아 LLM을 한 번 또는 고정된 방식으로 호출하고 결과를 반환하는
가장 단순한 형태의 LLM 애플리케이션이다. 실행 경로에 분기나 반복이 없다.

**Workflow**는 여러 처리 단계를 개발자가 미리 정한 순서대로 실행하는 구조이다.
각 단계에서 무엇을 할지는 고정되어 있으며, 입력 내용에 따라 순서가 바뀌지 않는다.

**Agent**는 현재 상태와 판단에 따라 다음에 무엇을 할지(어떤 Tool을 호출할지, 몇 번 반복할지)를  
LLM이 동적으로 결정하는 구조이다. 실행 경로가 요청마다 달라질 수 있다.  

### 3.2 개념이 필요한 이유

세 구조를 구분하지 않으면 설계 판단이 어려워진다. 
예를 들어 "오늘 날짜를 알려줘"처럼 단순한 요청에도 Tool 선택 로직을 갖춘 Agent를 적용하면 불필요한 LLM 호출과 지연시간이 늘어난다.    
반대로 "문서 종류에 따라 처리 방법이 달라져야 하는" 요청에 고정 Workflow만 사용하면 예외 상황을 처리하지 못한다.  
요청의 복잡도와 위험도에 맞는 구조를 선택하려면 세 구조의 실행 경로 차이를 먼저 이해해야 한다.  

### 3.3 주요 구성요소

| 구성요소 | 역할 |
|---|---|
| 사용자 요청(`user_request`) | 수행할 작업을 지정하는 입력 문자열 |
| 문서(`document`) | 세 방식이 공통으로 처리하는 대상 텍스트 |
| Tool 함수(`extract_keywords`, `summarize_document`) | 실제 작업을 수행하는 개별 기능 단위 |
| 실행 경로 결정 주체 | AI App은 없음, Workflow는 개발자, Agent는 LLM |
| `executed_steps` | 실제로 실행된 단계를 기록하는 로그 |

### 3.4 동작 과정

```text
[AI App]
사용자 요청 → LLM 1회 호출 → 답변

[Workflow]
사용자 요청 → 키워드 추출(고정) → 요약(고정) → 답변

[Agent]
사용자 요청 → 필요한 Tool 판단(LLM) → 선택된 Tool 실행(0개 이상) → 답변
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `run_ai_app()` | AI App: LLM 1회 호출 구조 |
| `run_fixed_workflow()` | Workflow: 고정 순서 실행 |
| `select_tools_with_agent()` | Agent: Tool 선택 판단 |
| `run_tool_agent()` | Agent: 선택된 Tool만 동적으로 실행 |
| `executed_steps` | 실제로 실행된 단계 기록 |

### 3.6 유사 개념과의 차이

| 기준 | AI App | Workflow | Agent |
|---|---|---|---|
| 실행 경로 | 단일 또는 단순 호출 | 개발자가 사전 정의 | 상태와 판단에 따라 변경 |
| Tool 선택 | 없음 또는 고정 | 개발자가 지정 | 모델(LLM)이 선택 |
| 자율성 | 낮음 | 중간 | 상대적으로 높음 |
| 통제 가능성 | 높음 | 높음 | 설계에 따라 달라짐 |
| 재현성 | 상대적으로 높음 | 높음 | 상대적으로 낮을 수 있음 |

### 3.7 사용 시점과 적용 조건

요청이 단순하고 처리 방법이 항상 같다면 AI App으로 충분하다.  
처리 단계는 여러 개이지만 순서와 방법이 고정되어 있다면 Workflow가 적합하다.  
입력에 따라 필요한 처리가 달라지고 그 판단 자체를 자동화해야 한다면 Agent 구조를 고려한다.  
Agent는 유연하지만 항상 더 좋은 선택은 아니다.  

### 3.8 한계와 주의사항

- Agent는 Tool 선택을 LLM에 맡기기 때문에, 입력 해석에 따라 실행 경로가 달라질 수 있는    
  구조적 위험을 가진다(Workflow처럼 코드로 경로를 고정하지 않는다). 이 실습처럼   
  `temperature=0`으로 고정하면 동일 입력에는 대부분 동일한 선택이 재현되지만, OpenAI API는   
  `temperature=0`에서도 100% 결정론적 재현을 보장하지는 않으므로 드물게 다른 결과가 나올 수 있다.  
  "재현성이 낮다"는 것은 실행할 때마다 무작위로 바뀐다는 뜻이 아니라, 코드가 경로를 보장하지 않는다는 뜻이다.  
- Agent는 Tool 선택 단계에서 추가 LLM 호출이 발생하므로 일반적으로 AI App이나 Workflow보다 지연시간과 비용이 늘어난다.
- Workflow는 고정된 순서를 강제하기 때문에 예상하지 못한 입력 유형에 대응하지 못한다.  

### 3.9 자주 발생하는 오해

Agent가 Workflow보다 항상 더 똑똑하거나 더 나은 결과를 낸다는 것은 오해이다.  
Workflow는 실행 순서가 고정되어 있어 결과가 누락될 위험이 없지만,   
Agent는 Tool 선택을 잘못하면 필요한 단계를 건너뛸 수 있다.  
이 문제는 9번 실패 실험과 10번 오류 수정 실습에서 함께 확인한다.

## 4. 실행 구조

이번 실습에서는 동일한 문서와 요청을 세 가지 함수로 각각 처리하고, 실행 로그를 비교한다.

```text
문서 + 사용자 요청
        │
        ├── run_ai_app()          → 단일 LLM 호출
        ├── run_fixed_workflow()  → 키워드 추출 → 요약 (고정 순서)
        └── run_tool_agent()      → Tool 선택(LLM) → 선택된 Tool 실행
        │
        ▼
   실행 로그 비교
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.  
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.  

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [25]:
# 1) 이 Notebook에서 사용할 공통 기능
from agentic_ai.config import get_settings
from agentic_ai.models import get_chat_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import DATA_DIR, OUTPUT_DIR, PROJECT_ROOT

# 2) 환경변수와 경로 확인
settings = get_settings()
print_environment_summary(settings, needs_chat_model=True)


[환경 설정 확인]
- 프로젝트: C:\Users\magpi\agentic_ai_lab_202607
- 데이터: C:\Users\magpi\agentic_ai_lab_202607\data
- 출력: C:\Users\magpi\agentic_ai_lab_202607\outputs
- OPENAI_API_KEY: 설정됨
- Chat Model: gpt-4.1-mini


## 6. 최소 실행 예제

LLM 호출 파이프라인이 정상 동작하는지 가장 단순한 질문으로 먼저 확인한다.

In [26]:
model = get_chat_model()
response = model.invoke("1+1은 얼마인가? 숫자만 답해줘.")
print(response.content)

2


## 7. 단계별 구현

실습에 사용할 문서를 불러온다.

In [27]:
document_path = DATA_DIR / "sample_report.txt"
document_text = document_path.read_text(encoding="utf-8")
print(document_text[:200], "...")

2026년 사내 리모트워크 운영 현황 보고서

작성일: 2026-01-15
작성부서: 인사운영팀
문서구분: 내부 보고서

1. 개요
본 보고서는 2025년 한 해 동안 시행된 리모트워크 제도의 운영 현황을 정리하고,
2026년도 제도 개선 방향을 제시하기 위해 작성되었다.

2. 운영 현황
2025년 기준 전체 임직원의 62%가 주 2회 이상 리모트워크를  ...


### 7.1 공통 Tool 함수 정의

`extract_keywords()`는 문서에서 핵심 키워드를 추출하고,  
`summarize_document()`는 문서를 2~3문장으로 요약한다.  
두 함수는 Workflow와 Agent 양쪽에서 재사용한다.  

In [28]:
def extract_keywords(document: str, n: int = 5) -> list[str]:
    """LLM을 이용해 문서에서 핵심 키워드 n개를 추출한다."""
    model = get_chat_model()
    prompt = (
        f"다음 문서에서 핵심 키워드 {n}개를 쉼표로만 구분해서 출력해줘. "
        f"다른 설명은 출력하지 마.\n\n문서:\n{document}"
    )
    response = model.invoke(prompt)
    keywords = [kw.strip() for kw in response.content.split(",") if kw.strip()]
    return keywords[:n]

keywords_preview = extract_keywords(document_text)
print(keywords_preview)

['리모트워크', '사용률', '만족도', '협업 지연', '온보딩']


**TODO**: `summarize_document()`를 `extract_keywords()`와 같은 패턴으로 작성한다.  
문서를 2~3문장으로 요약하는 프롬프트를 작성하고, `model.invoke()` 결과의 `.content`를 `strip()`해서 반환한다.  

예상 출력: 두세 문장으로 이루어진 한국어 요약 문자열

In [29]:
def summarize_document(document: str) -> str:
    """LLM을 이용해 문서를 2~3문장으로 요약한다."""
    model = get_chat_model()
    prompt = f"다음 문서를 2~3문장으로 요약해줘.\n\n문서:\n{document}"
    response = model.invoke(prompt)
    return response.content.strip()


summary_preview = summarize_document(document_text)
print(summary_preview)

2025년 사내 리모트워크 제도는 전체 임직원의 62%가 주 2회 이상 활용하며 높은 만족도를 보였으나, 협업 지연과 신입 직원 온보딩 문제 등이 나타났다. 2026년에는 팀별 필수 출근일 지정과 신입 직원의 사무실 근무 권장, 화상회의 시간 단축 등 개선 방안을 추진할 계획이다.


### 7.2 AI App 구현

요청과 문서를 합쳐 LLM을 한 번만 호출한다. 실행 단계는 항상 1개이다.

In [30]:
def run_ai_app(user_request: str, document: str) -> dict:
    """LLM을 한 번 호출해서 바로 답변을 생성한다."""
    # 계획이나 Tool 선택 없이 요청과 문서를 한 번의 프롬프트로 처리한다.
    model = get_chat_model()
    prompt = f"{user_request}\n\n문서:\n{document}"
    response = model.invoke(prompt)
    return {
        "mode": "ai_app",
        "executed_steps": ["single_llm_call"],
        "answer": response.content.strip(),
    }

### 7.3 Workflow 구현

`extract_keywords()` → `summarize_document()` 순서를 고정한다.  
사용자 요청 내용과 무관하게 항상 두 단계를 모두 실행한다.  

**TODO**: `run_fixed_workflow()`를 완성한다.  
`executed_steps` 리스트에 실행한 함수 이름을 순서대로 기록하고, 두 결과를 합쳐 `answer` 문자열을 만든다.  

In [31]:
def run_fixed_workflow(document: str) -> dict:
    """키워드 추출과 요약을 고정된 순서로 모두 실행한다."""
    executed_steps = []

    # 사용자 요청과 무관하게 정의된 두 단계를 항상 같은 순서로 실행한다.
    keywords = extract_keywords(document)
    executed_steps.append("extract_keywords")

    summary = summarize_document(document)
    executed_steps.append("summarize_document")

    answer = f"핵심 키워드: {', '.join(keywords)}\n요약: {summary}"
    return {
        "mode": "workflow",
        "executed_steps": executed_steps,
        "answer": answer,
    }

### 7.4 Agent 구현

Agent는 먼저 어떤 Tool이 필요한지 LLM에게 판단하게 한 뒤, 선택된 Tool만 실행한다.  
Workflow와 달리 요청 내용에 따라 실행되는 Tool의 종류와 개수가 달라질 수 있다.  

In [32]:
import json

# 툴 목록을 LLM이 선택할 수 있도록 정의한다.
AVAILABLE_TOOLS = {
    "extract_keywords": "문서에서 핵심 키워드를 추출한다.",
    "summarize_document": "문서를 짧게 요약한다.",
}


def select_tools_with_agent(user_request: str) -> list[str]:
    """사용자 요청을 처리하는 데 필요한 Tool 이름을 LLM이 판단하게 한다."""
    # get_chat_model()의 기본값은 temperature=0.0이므로, 동일한 user_request에는
    # 대부분 동일한 Tool 선택 결과가 재현된다. 다만 OpenAI API는 temperature=0에서도
    # 100% 결정론적 재현을 보장하지 않으므로, 드물게 다른 결과가 나올 수 있다.
    model = get_chat_model()
    tool_descriptions = "\n".join(
        f"- {name}: {desc}" for name, desc in AVAILABLE_TOOLS.items()
    )
    prompt = (
        "다음은 사용 가능한 Tool 목록이다.\n"
        f"{tool_descriptions}\n\n"
        f"사용자 요청: {user_request}\n\n"
        "이 요청을 처리하는 데 필요한 Tool 이름만 실행 순서대로 "
        "JSON 배열로 출력해줘. 예: [\"extract_keywords\", \"summarize_document\"] "
        "다른 설명은 출력하지 마."
    )

    # LLM은 실행할 함수가 아니라 Tool 이름 목록만 결정한다.
    response = model.invoke(prompt)

    # 모델 출력을 그대로 실행하지 않고 검증 파서를 통과시킨다.
    return parse_tool_selection(response.content)

**TODO**: `parse_tool_selection()`을 작성한다.   
LLM이 출력한 문자열을 `json.loads()`로 파싱하고, `AVAILABLE_TOOLS`에 실제로 존재하는 이름만 남긴다.   
JSON 파싱에 실패하면, 빈 리스트를 반환한다(오류를 발생시키지 않는다).  

예상 출력: `["extract_keywords", "summarize_document"]`와 같은 리스트, 또는 파싱 실패 시 `[]`

In [33]:
def parse_tool_selection(raw_text: str) -> list[str]:
    """LLM의 Tool 선택 JSON을 파싱하고, 유효한 이름을 순서대로 한 번씩만 반환한다."""
    try:
        tool_names = json.loads(raw_text.strip())
    except (json.JSONDecodeError, TypeError):
        return []
    if not isinstance(tool_names, list):
        return []

    # 허용 목록에 등록된 문자열만 남겨 임의의 이름이 실행되는 것을 막는다.
    valid_names = []
    for name in tool_names:
        if isinstance(name, str) and name in AVAILABLE_TOOLS and name not in valid_names:
            valid_names.append(name)
    return valid_names


In [34]:
def run_tool_agent(user_request: str, document: str) -> dict:
    """LLM이 선택한 Tool만 실행하여 답변을 생성한다."""
    selected_tools = select_tools_with_agent(user_request)
    executed_steps = []
    results = {}

    # 선택된 Tool 이름을 실제 Python 함수에 연결한다.
    for tool_name in selected_tools:
        if tool_name == "extract_keywords":
            results["keywords"] = extract_keywords(document)
        elif tool_name == "summarize_document":
            results["summary"] = summarize_document(document)
        executed_steps.append(tool_name)

    # 실제로 실행된 Tool의 결과만 최종 답변에 포함한다.
    parts = []
    if "keywords" in results:
        parts.append(f"핵심 키워드: {', '.join(results['keywords'])}")
    if "summary" in results:
        parts.append(f"요약: {results['summary']}")
    answer = "\n".join(parts) if parts else "선택된 Tool이 없어 처리할 수 없습니다."

    return {
        "mode": "agent",
        "selected_tools": selected_tools,
        "executed_steps": executed_steps,
        "answer": answer,
    }

In [35]:
request = "입력된 문서에서 핵심 키워드를 추출하고 요약해줘."

# 동일한 입력으로 AI App, 고정 Workflow, Agent의 실행 경로를 비교한다.
ai_app_result = run_ai_app(request, document_text)
workflow_result = run_fixed_workflow(document_text)
agent_result = run_tool_agent(request, document_text)

for result in (ai_app_result, workflow_result, agent_result):
    print(f"\n[{result['mode']}] executed_steps = {result['executed_steps']}")
    print(result["answer"])


[ai_app] executed_steps = ['single_llm_call']
### 핵심 키워드
- 리모트워크 운영 현황
- 2025년 리모트워크 사용률 (전체 62%, 개발팀 81%, 영업팀 24%)
- 사내 메신저 사용 시간 증가 (18%)
- 만족도 (74% 만족)
- 불만족 사유: 협업 지연, 화상회의 피로도, 장비 지원 부족
- 정보 격차 문제
- 신입 직원 온보딩 어려움
- 2026년 개선 계획: 팀별 필수 출근일 지정, 신입 주 3회 이상 출근 권장, 화상회의 시간 단축(30분 가이드라인)
- 협업 방식 및 온보딩 절차 보완 필요

### 요약
2025년 한 해 동안 전체 임직원의 62%가 주 2회 이상 리모트워크를 활용했으며, 특히 개발팀의 사용률이 81%로 가장 높았다. 리모트워크 도입으로 사내 메신저 사용 시간이 18% 증가했고, 74%의 임직원이 제도에 만족했다. 다만 협업 지연, 화상회의 피로도, 장비 지원 부족 등의 불만이 있었고, 일부 부서에서는 리모트워크와 사무실 근무자 간 정보 격차가 발생했다. 신입 직원들은 온보딩 과정에서 어려움을 겪는 것으로 나타났다. 2026년에는 팀별 필수 출근일 지정과 신입 직원의 주 3회 이상 사무실 근무 권장, 화상회의 시간을 30분으로 단축하는 가이드라인 도입 등 개선 방안을 추진할 계획이다. 전반적으로 리모트워크 제도는 긍정적 효과를 보였으나 협업과 온보딩 절차 보완이 필요하다.

[workflow] executed_steps = ['extract_keywords', 'summarize_document']
핵심 키워드: 리모트워크, 사용률, 만족도, 협업 지연, 온보딩
요약: 2025년 기준 전체 임직원의 62%가 주 2회 이상 리모트워크를 활용했으며, 74%가 제도에 만족하는 것으로 나타났다. 다만 협업 지연과 신입 직원 온보딩 어려움 등의 문제점이 있어, 2026년에는 필수 출근일 지정과 화상회의 시간 단축 등 개선 방안을 도입할 계획이다.

[agent] executed_steps = 

## 8. 실행 결과 관찰

필수 실습 요청으로 세 방식을 모두 실행하고, `executed_steps`(실행된 단계)와
`selected_tools`(Agent가 선택한 Tool)를 비교한다.

**결과 해석**:  
- AI App은 `executed_steps`가 항상 1개(`single_llm_call`)로 고정되어 있어 내부적으로 어떤 처리를 했는지 알 수 없다.
- Workflow는 요청 내용과 무관하게 항상 `["extract_keywords", "summarize_document"]` 두 단계를 실행한다. 
- Agent는 `selected_tools` 값이 요청 내용에 따라 달라질 수 있으며, 이번 요청처럼 두 기능이 모두 필요한 경우 Workflow와 유사한 결과를 낼 수 있다.

### 8.1 AI App 실행 흐름

AI App은 요청과 문서를 하나의 프롬프트로 합친 뒤 LLM을 **한 번만 호출**한다. 분기, 반복, Tool 선택 과정은 없다.

```mermaid
sequenceDiagram
    autonumber
    actor U as 사용자
    participant A as run_ai_app()
    participant L as Chat Model (LLM)

    U->>A: user_request + document 전달
    activate A
    A->>A: 요청과 문서를 하나의 prompt로 결합
    A->>L: model.invoke(prompt)
    activate L
    L-->>A: response.content 반환
    deactivate L
    A->>A: 응답 정리 및 결과 dict 생성
    Note over A: executed_steps = [single_llm_call]
    A-->>U: mode, executed_steps, answer 반환
    deactivate A
```

> **관찰 포인트:** 실행 경로가 `사용자 → run_ai_app() → LLM → 사용자`로 단순하며, 내부 처리 과정은 `single_llm_call` 하나로만 기록된다.

### 8.2 Workflow 실행 흐름

Workflow는 개발자가 정한 순서에 따라 **키워드 추출 → 문서 요약**을 항상 차례대로 실행한다. 각 단계에서 LLM을 한 번씩 호출하므로 전체 LLM 호출은 2회이다.

```mermaid
sequenceDiagram
    autonumber
    actor U as 사용자
    participant W as run_fixed_workflow()
    participant K as extract_keywords()
    participant S as summarize_document()
    participant L as Chat Model (LLM)

    U->>W: document 전달
    activate W
    Note over W,S: 실행 순서는 개발자가 미리 고정

    W->>K: 키워드 추출 요청
    activate K
    K->>L: model.invoke(키워드 추출 prompt)
    activate L
    L-->>K: 키워드 응답
    deactivate L
    K-->>W: keywords 반환
    deactivate K
    W->>W: executed_steps에 extract_keywords 기록

    W->>S: 문서 요약 요청
    activate S
    S->>L: model.invoke(요약 prompt)
    activate L
    L-->>S: 요약 응답
    deactivate L
    S-->>W: summary 반환
    deactivate S
    W->>W: executed_steps에 summarize_document 기록
    W->>W: keywords와 summary를 최종 answer로 결합
    W-->>U: mode, executed_steps, answer 반환
    deactivate W
```

> **관찰 포인트:** 사용자 요청의 내용과 관계없이 `extract_keywords`와 `summarize_document`가 항상 같은 순서로 실행되어 경로를 예측하기 쉽다.

### 8.3 AI Agent 실행 흐름

AI Agent는 먼저 LLM에게 필요한 Tool을 선택하게 하고, 선택된 Tool만 순서대로 실행한다. 따라서 사용자 요청에 따라 실행 경로와 LLM 호출 횟수가 달라질 수 있다.

```mermaid
sequenceDiagram
    autonumber
    actor U as 사용자
    participant A as run_tool_agent()
    participant T as select_tools_with_agent()
    participant K as extract_keywords()
    participant S as summarize_document()
    participant L as Chat Model (LLM)

    U->>A: user_request + document 전달
    activate A
    A->>T: user_request 전달
    activate T
    T->>L: model.invoke(Tool 선택 prompt)
    activate L
    L-->>T: Tool 이름 JSON 배열 반환
    deactivate L
    T->>T: 응답 파싱 및 유효한 Tool만 필터링
    T-->>A: selected_tools 반환
    deactivate T
    Note over A,T: 요청에 따라 선택 결과가 달라짐

    loop selected_tools의 각 Tool
        alt extract_keywords 선택
            A->>K: document 전달
            activate K
            K->>L: model.invoke(키워드 추출 prompt)
            activate L
            L-->>K: 키워드 응답
            deactivate L
            K-->>A: keywords 반환
            deactivate K
        else summarize_document 선택
            A->>S: document 전달
            activate S
            S->>L: model.invoke(요약 prompt)
            activate L
            L-->>S: 요약 응답
            deactivate L
            S-->>A: summary 반환
            deactivate S
        end
        A->>A: 실행한 Tool을 executed_steps에 기록
    end

    alt 하나 이상의 Tool 실행
        A->>A: 실행 결과를 최종 answer로 결합
    else 선택된 Tool이 없음
        A->>A: 처리 불가 메시지 생성
    end
    A-->>U: selected_tools, executed_steps, answer 반환
    deactivate A
```

> **관찰 포인트:** 이번 요청에서 두 Tool이 모두 선택되면 Tool 선택 1회, 키워드 추출 1회, 요약 1회로 LLM을 총 3회 호출한다. 그러나 선택 결과가 달라지면 실행 단계와 호출 횟수도 달라진다.

## 9. 실패 실험: 모호한 요청에서의 Tool 선택 누락

이번에는 모호한 요청을 Agent에게 준다.  
Workflow는 요청과 무관하게 항상 두 단계를 모두 실행하지만,  
**Agent**는 **Tool 선택을 LLM의 판단에 맡기기 때문**에 모호한 요청에서는 필요한 Tool 중 일부를 판단 단계에서부터 누락할 수 있다.  
`select_tools_with_agent()`는 `temperature=0.0`으로 호출되므로 같은 요청에 대해서는 대부분 같은 결과가 재현된다.   
(다만 OpenAI API는 `temperature=0`에서도 100% 결정론적 재현을 보장하지 않으므로, 드물게 다른 값이 나올 수 있다).   
이 실습에서 관찰하려는 실패는 "매번 달라지는 무작위성" 때문이 아니라,    
"이 요청을 이렇게 해석하면 하나의 Tool을 놓친다"는 판단 오류이며 대부분의 경우 재현 가능하다.    
10번에서는 선택이 완전히 비어 있는 경우까지 강제로 재현해 본다.  

In [36]:
# 의도가 모호할 때 Agent가 Tool을 선택하지 못하는 실패 사례를 관찰한다.
vague_request = "이 자료 어때?"

vague_result = run_tool_agent(vague_request, document_text)
print("selected_tools:", vague_result["selected_tools"])
print("answer:", vague_result["answer"])

selected_tools: ['summarize_document']
answer: 요약: 2025년 사내 리모트워크 제도는 전체 임직원의 62%가 주 2회 이상 활용하며 높은 만족도를 보였으나, 협업 지연과 신입 직원 온보딩 문제 등 일부 과제가 발견되었다. 2026년에는 팀별 필수 출근일 지정과 신입 직원의 사무실 근무 권장, 화상회의 시간 단축 등의 개선 방안이 추진될 예정이다.


### 9.1 실행 시퀀스 다이어그램

아래 다이어그램은 현재 실행에서 모호한 요청 `이 자료 어때?`를 요약 요청으로 해석하여 `summarize_document`만 선택한 사례를 보여준다.

```mermaid
sequenceDiagram
    autonumber
    actor U as 사용자
    participant A as run_tool_agent()
    participant T as select_tools_with_agent()
    participant S as summarize_document()
    participant L as Chat Model (LLM)

    U->>A: 모호한 요청 + document 전달
    Note over U,A: user_request = 이 자료 어때?
    activate A
    A->>T: user_request 전달
    activate T
    T->>L: model.invoke(Tool 선택 prompt)
    activate L
    L-->>T: [summarize_document]
    deactivate L
    T->>T: JSON 파싱 및 Tool 이름 검증
    T-->>A: selected_tools 반환
    deactivate T
    Note over A: extract_keywords는 선택되지 않아 실행하지 않음

    A->>S: document 전달
    activate S
    S->>L: model.invoke(요약 prompt)
    activate L
    L-->>S: 문서 요약 응답
    deactivate L
    S-->>A: summary 반환
    deactivate S
    A->>A: executed_steps에 summarize_document 기록
    A->>A: 요약 결과로 answer 생성
    A-->>U: selected_tools, executed_steps, answer 반환
    deactivate A
```

> **관찰 포인트:** Agent가 Tool을 전혀 선택하지 못한 것은 아니다.   
> 모호한 요청을 요약 요청으로 해석하면서 개발자가 기대할 수 있는 `extract_keywords` 경로가 실행되지 않은 사례이다.   
> 모델을 다시 실행하면 선택 결과가 달라질 수 있다.  

**원인 분석 질문**

- `selected_tools`가 비어 있거나 예상과 다르다면, `select_tools_with_agent()`가 LLM에게
  보낸 프롬프트와 실제로 돌아온 응답은 무엇이었는가?
- `parse_tool_selection()`은 이런 경우를 어떻게 처리하고 있는가?
- Workflow였다면 이 요청에 대해 어떤 결과가 나왔을까?
- 이 셀을 다시 실행해도 같은 결과가 나오는 이유는 무엇인가? (`get_chat_model()`의
  `temperature` 기본값을 확인해본다.)

아래 셀에서 `select_tools_with_agent()`가 반환한 원문 응답을 직접 확인해서 원인을 살펴본다.

In [ ]:
model = get_chat_model()
raw_response = model.invoke(
    f"""다음은 사용 가능한 Tool 목록이다.
    - extract_keywords: 문서에서 핵심 키워드를 추출한다.
    - summarize_document: 문서를 짧게 요약한다.

    사용자 요청: {vague_request}
    이 요청을 처리하는 데 필요한 Tool 이름만 실행 순서대로 
    JSON 배열로 출력해줘. 다른 설명은 출력하지 마."""
)
print(raw_response.content)
print(repr(raw_response.content))   # representation : 객체의 값을 표현한 문자열

["summarize_document"]
'["summarize_document"]'


## 10. 오류 수정 실습

Agent가 Tool을 선택하지 못했을 때 무조건 모든 Tool을 실행하면, 쓰기·전송 Tool이 추가된 환경에서 위험할 수 있다.  
이 실습에서는 읽기 전용인 `summarize_document` 하나만 안전한 기본값으로 사용한다.

**TODO**: 선택 함수를 주입받는 `select_tools_with_fallback()`을 작성하고,  
`empty_selector`로 빈 선택을 강제해 fallback(차선책)이 실제로 실행되는지 검증한다.


In [46]:
# 빈 선택을 결정적으로 재현할 수 있도록 selector를 주입받는다.
def select_tools_with_fallback(user_request: str, selector=select_tools_with_agent) -> tuple[list[str], bool]:
    """선택 결과가 비면 읽기 전용인 summarize_document만 안전한 기본값으로 사용한다."""
    tools = selector(user_request)
    # 선택된 Tool이 하나도 없을 때만 fallback을 사용한다(정상 선택 결과는 그대로 둔다).
    fallback_used = not tools
    if fallback_used:
        # 쓰기·전송 등 부수효과가 있는 Tool이 아니라, 읽기 전용 Tool 하나만 기본값으로 삼아
        # "아무것도 못 함" 대신 안전하게 최소한의 답변을 낼 수 있게 한다.
        tools = ["summarize_document"]
    return tools, fallback_used


def run_tool_agent_v2(user_request: str, document: str, selector=select_tools_with_agent) -> dict:
    """7번의 run_tool_agent()에 fallback 로직을 추가한 수정 버전."""
    # 원래의 select_tools_with_agent() 대신 fallback을 거친 선택 결과를 사용한다.
    selected_tools, fallback_used = select_tools_with_fallback(user_request, selector)
    executed_steps = []
    results = {}
    for tool_name in selected_tools:
        if tool_name == "extract_keywords":
            results["keywords"] = extract_keywords(document)
            executed_steps.append(tool_name)
        elif tool_name == "summarize_document":
            results["summary"] = summarize_document(document)
            executed_steps.append(tool_name)

    parts = []
    if "keywords" in results:
        parts.append(f"핵심 키워드: {', '.join(results['keywords'])}")
    if "summary" in results:
        parts.append(f"요약: {results['summary']}")
    answer = "\n".join(parts) if parts else "실행 가능한 Tool이 없어 처리할 수 없습니다."

    return {
        "mode": "agent_v2",
        "selected_tools": selected_tools,
        "executed_steps": executed_steps,
        # fallback_used를 함께 반환해 이번 답변이 정상 선택 결과인지,
        # 빈 선택을 기본값으로 대체한 결과인지 로그에서 구분할 수 있게 한다.
        "fallback_used": fallback_used,
        "answer": answer,
    }


def empty_selector(_: str) -> list[str]:
    """fallback 실험을 위한 결정적 실패 주입 함수."""
    # LLM 호출 없이 항상 빈 리스트를 반환해, 9번처럼 우연히 재현되는 실패가 아니라
    # "선택이 완전히 비어 있는 경우"를 매번 동일하게 강제로 만든다.
    return []


# select_tools_with_agent() 대신 empty_selector를 주입해 빈 선택 상황을 결정적으로 재현한다.
fixed_result = run_tool_agent_v2(vague_request, document_text, selector=empty_selector)
print("fallback_used:", fixed_result["fallback_used"])
print("selected_tools:", fixed_result["selected_tools"])
print("answer:", fixed_result["answer"])


fallback_used: True
selected_tools: ['summarize_document']
answer: 요약: 2025년 사내 리모트워크 제도는 전체 임직원의 62%가 주 2회 이상 활용하며 높은 만족도를 보였으나, 협업 지연과 신입 직원 온보딩 문제 등이 나타났다. 2026년에는 팀별 필수 출근일 지정과 신입 직원의 사무실 근무 권장, 화상회의 시간 단축 등 개선 방안을 추진할 계획이다.


### 10.1 수정 코드 실행 시퀀스 다이어그램

아래 다이어그램은 `empty_selector`로 빈 Tool 선택을 강제하고, `select_tools_with_fallback`이 안전한 기본 Tool인 `summarize_document`를 적용하는 과정을 보여준다.

```mermaid
sequenceDiagram
    autonumber
    actor U as 사용자 또는 실행 셀
    participant A as run_tool_agent_v2()
    participant F as select_tools_with_fallback()
    participant E as empty_selector()
    participant S as summarize_document()
    participant L as Chat Model (LLM)

    U->>A: vague_request + document + empty_selector 전달
    activate A
    A->>F: user_request + selector 전달
    activate F
    F->>E: selector(user_request) 호출
    activate E
    E-->>F: 빈 리스트 [] 반환
    deactivate E

    alt 선택된 Tool이 없음
        F->>F: fallback_used = True
        F->>F: tools = [summarize_document]
    else Tool 선택 성공
        F->>F: fallback_used = False
    end
    F-->>A: selected_tools + fallback_used 반환
    deactivate F
    Note over A,E: empty_selector 사용 시 Tool 선택용 LLM 호출 없음

    A->>S: document 전달
    activate S
    S->>L: model.invoke(요약 prompt)
    activate L
    L-->>S: 문서 요약 응답
    deactivate L
    S-->>A: summary 반환
    deactivate S
    A->>A: executed_steps에 summarize_document 기록
    A->>A: 요약 결과로 answer 생성
    A-->>U: selected_tools, executed_steps, fallback_used, answer 반환
    deactivate A
```

> **관찰 포인트:**   
> 빈 선택이 발생해도 모든 Tool을 무조건 실행하지 않고, 읽기 전용인 `summarize_document` 하나만 안전한 기본값으로 실행한다.   
> 따라서 `fallback_used=True`로 기록되며 실패 상황도 매번 동일하게 재현된다.  

**수정 결과 재검증**: `fallback_used=True`이고, 선택 및 실제 실행 Tool이
`["summarize_document"]`와 정확히 일치해야 한다.


In [39]:
# empty_selector로 빈 선택을 강제했으니 fallback이 실제로 발동했는지 확인한다.
assert fixed_result["fallback_used"] is True

# fallback으로 대체된 Tool이 의도한 대로 summarize_document 하나뿐인지 확인한다.
assert fixed_result["selected_tools"] == ["summarize_document"]

# 선택된 Tool과 실제로 실행된 Tool이 일치하는지 확인한다(선택만 되고 실행은 안 되는 경우를 방지).
assert fixed_result["executed_steps"] == ["summarize_document"]

# fallback 이전처럼 "처리할 수 없습니다" 응답으로 빠지지 않고 실제 답변이 만들어졌는지 확인한다.
assert fixed_result["answer"] != "실행 가능한 Tool이 없어 처리할 수 없습니다."
print("재검증 통과")


재검증 통과


## 11. 도전 과제

1. `AVAILABLE_TOOLS`에 새로운 Tool(예: `translate_document`)을 설명만 추가하고,
   Agent가 이 Tool을 실제로 선택하는지 관찰해본다(함수 구현은 하지 않아도 된다).
2. Workflow에 세 번째 고정 단계를 추가했을 때 `executed_steps`가 어떻게 바뀌는지 확인한다.
3. "이 문서를 요약만 해줘."처럼 한 가지 기능만 필요한 요청을 만들어 Agent와 Workflow의
   `executed_steps` 차이를 비교한다.

In [40]:
# 도전 과제 1: AVAILABLE_TOOLS에 새로운 Tool 설명만 추가하고, Agent가 이 Tool을 선택하는지 관찰한다.
# (translate_document 함수는 구현하지 않는다 — Tool 선택 결과만 확인하는 것이 목적이다.)
AVAILABLE_TOOLS["translate_document"] = "문서를 영어로 번역한다."

translate_request = "이 문서를 영어로 번역해줘."
translate_selected = select_tools_with_agent(translate_request)
print("선택된 Tool:", translate_selected)

# 이후 셀(run_tool_agent 등)에 영향을 주지 않도록 실험이 끝나면 원상 복구한다.
del AVAILABLE_TOOLS["translate_document"]

선택된 Tool: ['translate_document']


In [41]:
# 도전 과제 2: Workflow에 세 번째 고정 단계를 추가했을 때 executed_steps가 어떻게 바뀌는지 확인한다.
def count_words(document: str) -> int:
    """문서의 단어 수를 세는 세 번째 고정 단계(LLM 호출 없음)."""
    return len(document.split())


def run_fixed_workflow_v2(document: str) -> dict:
    """run_fixed_workflow()에 count_words 단계를 추가한 버전."""
    executed_steps = []

    keywords = extract_keywords(document)
    executed_steps.append("extract_keywords")

    summary = summarize_document(document)
    executed_steps.append("summarize_document")

    word_count = count_words(document)
    executed_steps.append("count_words")

    answer = f"핵심 키워드: {', '.join(keywords)}\n요약: {summary}\n단어 수: {word_count}"
    return {
        "mode": "workflow_v2",
        "executed_steps": executed_steps,
        "answer": answer,
    }


workflow_v2_result = run_fixed_workflow_v2(document_text)
print("executed_steps:", workflow_v2_result["executed_steps"])

executed_steps: ['extract_keywords', 'summarize_document', 'count_words']


In [42]:
# 도전 과제 3: 한 가지 기능만 필요한 요청에서 Agent와 Workflow의 executed_steps 차이를 비교한다.
summary_only_request = "이 문서를 요약만 해줘."
summary_only_agent_result = run_tool_agent(summary_only_request, document_text)

print("Agent executed_steps:", summary_only_agent_result["executed_steps"])

print("Workflow executed_steps:", workflow_result["executed_steps"])

Agent executed_steps: ['summarize_document']
Workflow executed_steps: ['extract_keywords', 'summarize_document']


## 12. 테스트

**테스트 유형: 단위 테스트 — 결정적, 외부 API 호출 없음**(assert문 사용)  

LLM 호출 없이 확인할 수 있는 순수 로직(`parse_tool_selection`)을 검증한다.  
  
assert문 문법: 
`assert 조건, "오류 메시지"`

In [43]:
# 정상 케이스: 유효한 Tool 이름 두 개가 순서대로 담긴 JSON은 그대로 리스트로 반환되어야 한다.
assert parse_tool_selection('["extract_keywords", "summarize_document"]') == [
    "extract_keywords", "summarize_document"
]
# 허용 목록 필터링: AVAILABLE_TOOLS에 없는 이름("unknown_tool")은 제거되고 유효한 이름만 남아야 한다.
assert parse_tool_selection('["extract_keywords", "unknown_tool"]') == ["extract_keywords"]
# JSON 파싱 실패: JSON 형식이 아닌 문자열은 예외를 던지지 않고 빈 리스트를 반환해야 한다.
assert parse_tool_selection("이건 JSON이 아닙니다") == []
# 타입 불일치: 최상위 값이 리스트가 아니라 단일 문자열이면 빈 리스트를 반환해야 한다.
assert parse_tool_selection('"단일 문자열"') == []
# 타입 불일치: 리스트 안에 문자열이 아닌 다른 리스트가 들어 있으면 해당 항목은 무시되어 빈 리스트가 되어야 한다.
assert parse_tool_selection('[["extract_keywords"]]') == []
# 중복 제거: 같은 Tool 이름이 여러 번 나와도 한 번만 남아야 한다.
assert parse_tool_selection('["extract_keywords", "extract_keywords"]') == ["extract_keywords"]
print("parse_tool_selection 테스트 통과")

parse_tool_selection 테스트 통과


## 13. 결과 저장

세 방식의 실행 로그를 `outputs/logs/01-1_comparison_log.json`에 저장한다.

In [44]:
from agentic_ai.logging_utils import save_log

# 정상 비교와 fallback 전후 결과를 함께 저장해 실험을 재현할 수 있게 한다.
comparison_log = {
    "request": request,
    "ai_app": ai_app_result,
    "workflow": workflow_result,
    "agent": agent_result,
    "failure_experiment": {
        "vague_request": vague_request,
        "before_fallback": vague_result,
        "after_fallback": fixed_result,
    },
}
saved_path = save_log(comparison_log, OUTPUT_DIR / "logs" / "01-1_comparison_log.json")
print("저장 위치:", saved_path)

저장 위치: C:\Users\magpi\agentic_ai_lab_202607\outputs\logs\01-1_comparison_log.json


## 14. 핵심 정리

- AI App, Workflow, Agent는 실행 경로를 누가, 어떻게 결정하는지에 따라 구분된다.
- Workflow는 실행 순서가 고정되어 있어 재현성이 높지만 유연성이 낮다.
- Agent는 Tool 선택을 동적으로 판단하므로 유연하지만, 판단이 틀리면 필요한 단계를
  건너뛸 수 있어 Workflow보다 항상 우수하다고 할 수 없다.
- `executed_steps`와 `selected_tools`를 로그로 남기면 각 방식의 실행 경로 차이를
  객관적으로 비교할 수 있다.

## 15. 확인 문제

1. AI App, Workflow, Agent 중 자율성이 가장 낮은 구조는 무엇이며 그 이유는 무엇인가?   
2. Workflow가 Agent보다 재현성이 높은 이유를 실행 경로 관점에서 설명하시오.   
3. 9번 실패 실험에서 Agent가 `extract_keywords`를 선택하지 못하고 `summarize_document`만 선택한 원인은 무엇이었는가?  
   이 결과와 10번에서 `empty_selector`로 재현한 "완전히 빈 선택" 상황은 어떻게 다른가?  
4. 모든 요청에 Agent 구조를 적용하는 것이 항상 바람직하지 않은 이유를 서술하시오.  